In [16]:
import pandas as pd

# Load the dataset sheets to understand its structure
excel_file = '/content/campus_event_analytics_dataset (3).xlsx'
xls = pd.ExcelFile(excel_file)
print("Sheet names:", xls.sheet_names)

# Load the first sheet into a DataFrame
df = pd.read_excel(excel_file, sheet_name=0)
display(df.head())
display(df.info())

Sheet names: ['Campus Events Data', 'KPI Summary']


,Event ID,Event Name,Category,Department,Event Date,Venue,Registered Students,Attended Students,Turnout Rate,Satisfaction Rating
0,EVT-1001,Data Science with Python,Workshop,EEE,2025-08-21,Computer Lab 3,142,110,0.774648,3.6
1,EVT-1002,Table Tennis Cup,Sports,Mechanical,2025-12-28,Indoor Sports Complex,232,212,0.913793,4.7
2,EVT-1003,Drama & Theatre Play,Cultural,Civil,2025-08-22,Main Auditorium,585,498,0.851282,4.2
3,EVT-1004,Drama & Theatre Play,Cultural,Mechanical,2025-09-24,Indoor Sports Complex,593,542,0.913997,3.6
4,EVT-1005,Table Tennis Cup,Sports,CSE,2025-10-28,Open Air Theatre,239,207,0.866109,4.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Event ID             1001 non-null   object 
 1   Event Name           1000 non-null   object 
 2   Category             1000 non-null   object 
 3   Department           1000 non-null   object 
 4   Event Date           1000 non-null   object 
 5   Venue                1000 non-null   object 
 6   Registered Students  1001 non-null   int64  
 7   Attended Students    1001 non-null   int64  
 8   Turnout Rate         1001 non-null   float64
 9   Satisfaction Rating  1001 non-null   float64
dtypes: float64(2), int64(2), object(6)
memory usage: 78.3+ KB


None

In [17]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# --- Step 0: Data Preprocessing & Cleaning ---
# Reload and parse date, replace missing values as specified in instructions mapped to our campus dataset
df = pd.read_excel('/content/campus_event_analytics_dataset (3).xlsx', sheet_name='Campus Events Data')

# Ensure clean strings and fill missing categories/departments/venues with 'none'
df['Category'] = df['Category'].fillna('none').astype(str)
df['Department'] = df['Department'].fillna('none').astype(str)
df['Venue'] = df['Venue'].fillna('none').astype(str)
df['Event Name'] = df['Event Name'].fillna('none').astype(str)

# Convert Event Date to datetime, drop or fill NaT
df['Event Date'] = pd.to_datetime(df['Event Date'], errors='coerce')
# Fill missing dates with a placeholder or drop them; we'll drop rows where Event Date is null for cleaner time animations
df = df.dropna(subset=['Event Date'])
# Extract Year for the animation step
df['Year'] = df['Event Date'].dt.year

print(f"Data cleaned. Rows available for visualization: {len(df)}")

Data cleaned. Rows available for visualization: 1000


In [18]:
import plotly.express as px

fig1 = px.histogram(
    df,
    x="Registered Students",
    nbins=30,
    title="Task 1.1: Distribution of Registered Students"
)
fig1.update_layout(xaxis_rangeslider_visible=True)
fig1.show()

In [19]:
import plotly.express as px

fig2 = px.scatter(
    df,
    x="Registered Students",
    y="Attended Students",
    color="Category",
    log_y=True,
    hover_data=["Event Name", "Department", "Event Date"],
    title="Task 1.2: Registered vs Attended Students colored by Category (log_y=True)"
)
fig2.show()

In [20]:
import plotly.express as px

fig3 = px.box(
    df,
    x="Category",
    y="Registered Students",
    color="Category",
    points="all",
    title="Task 1.3: Registered Students by Category (with individual points)"
)
fig3.show()

In [21]:
import plotly.express as px

fig4 = px.scatter_3d(
    df,
    x="Registered Students",
    y="Attended Students",
    z="Satisfaction Rating",
    color="Category",
    title="Task 1.4: 3D Scatter Plot of Event Metrics"
)
fig4.show()

In [22]:
import plotly.express as px

# Create a copy sorted by year and establish sizing parameters
anim_df = df.sort_values(by='Year').copy()
anim_df['Size_Ref'] = anim_df['Satisfaction Rating'].clip(lower=0.1)

fig5 = px.scatter(
    anim_df,
    x="Registered Students",
    y="Attended Students",
    animation_frame="Year",
    animation_group="Event ID",
    size="Size_Ref",
    color="Category",
    hover_name="Event Name",
    range_x=[df['Registered Students'].min() - 10, df['Registered Students'].max() + 10],
    range_y=[df['Attended Students'].min() - 10, df['Attended Students'].max() + 10],
    title="Task 2.1: Yearly Progression of Event Attendance (Size scaled by Satisfaction Rating)"
)
fig5.show()

In [23]:
import plotly.graph_objects as go

metrics = ["Attended Students", "Turnout Rate", "Satisfaction Rating"]
fig6 = go.Figure()

for i, metric in enumerate(metrics):
    fig6.add_trace(
        go.Scatter(
            x=df["Registered Students"],
            y=df[metric],
            mode='markers',
            marker=dict(color=df['Satisfaction Rating'] if metric == 'Turnout Rate' else i * 50, colorscale='Viridis', showscale=True),
            text=df['Event Name'],
            hovertemplate='Registered: %{x}<br>Value: %{y}<br>Event: %{text}',
            name=metric,
            visible=(i == 0)
        )
    )

buttons = []
for i, metric in enumerate(metrics):
    visible_list = [False] * len(metrics)
    visible_list[i] = True

    buttons.append(dict(
        label=metric,
        method="update",
        args=[{"visible": visible_list},
              {"yaxis": {"title": metric}}]
    ))

fig6.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            x=0.15,
            xanchor="left",
            y=1.15,
            yanchor="top"
        )
    ],
    xaxis_title="Registered Students",
    yaxis_title=metrics[0],
    title="Task 2.2: Interactive Scatter Plot with Dynamic Y-Axis Metric"
)
fig6.show()